# Train the “Svegliati Ara” wake phrase for the Ara shell

This notebook trains an [openWakeWord](https://github.com/dscripka/openWakeWord)
keyword model for the phrase **“Svegliati ara”** and gives you a single
`svegliati_ara.onnx` file to load into the Ara shell. It follows openWakeWord's
automatic training recipe (synthetic speech, room reverb, background noise,
~2,000 hours of precomputed negative audio), adapted to the current versions of
the speech generator and of Colab's libraries.

The phrase is Italian. Half of the clips come from native Italian Piper voices
(`it_IT-paola-medium`, `it_IT-riccardo-x_low`), the other half from the English
multi-speaker generator (904 voices, blended into new ones) for speaker
variety. The English generator is not given an English spelling: the phrase is
phonemized by espeak-ng in Italian (`zvˈeʎatɪ ˈaɾa`) and mapped onto the
nearest phonemes the English voices know (`zvˈɛljɑtɪ ˈɑɾɑ`: ʎ → lj, a → ɑ,
e → ɛ; stress and the tapped r are kept). Any English respelling loses the
stress pattern or the tap: espeak reads “zvel yah tee ah rah” as
`zˈiːvˈɛl jˈɑː tˈiːʲ ˈɑː ɹˈɑː` (a spelled-out “Z”, every syllable stressed,
English r), and “svell yah tee ah rah” as `svˈɛl jˈɑː tˈiːʲ ˈɑː ɹˈɑː`. Listen
to the samples after the generation step to check both halves.

Two voices give limited variety, and they run on the CPU (about ten times
slower than the GPU generator), so only a fifth of their clips are actually
synthesized and the rest are copies that the augmentation step turns into
different examples (noise, reverb, pitch, equalizer).

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then
run the cells from top to bottom (Runtime → Run all works too). Keep the browser
tab open; a free Colab session ends after about 90 minutes of inactivity or
12 hours in total.

| Step | Roughly |
|---|---|
| 1–2. Settings, installation | 3–5 min |
| 3. Data download (~20 GB) | 10–15 min |
| 4. Generate speech clips | 30–45 min |
| 5. Augment clips, compute features | 10–15 min |
| 6. Train | 15–25 min |
| 7. Check and download | 1 min |

Every step skips work that is already done. After an interruption (or a
session restart) simply run all cells again from the top: finished steps take
seconds, and the cut-off step continues where it stopped. To train from
scratch, delete `/content/oww/output` (or start a new session).

## 1. Settings

In [ ]:
# ===== Phrase settings: the only cell you may want to edit =====

MODEL_NAME = "svegliati_ara"        # file name of the result: svegliati_ara.onnx
TARGET_PHRASE = "svegliati ara"
PHRASE_LANGUAGE = "it"              # espeak language the English voices use for the phrase

# Phrases that must NOT wake Ara, grouped by the language they are spoken in.
# openWakeWord adds thousands of automatically found sound-alikes on top.
CUSTOM_NEGATIVES = {
    "it": ["sveglia", "svegliati", "sveglia ragazzi", "Ara", "Sarah", "Laura",
           "Clara", "Chiara", "vegliate", "sveglio", "svelati", "è ora", "va bene",
           "Svizzera", "sbagliati", "sbagliato", "svegliatevi", "si sveglia",
           "sveglia amore", "ciao Sara", "dai Sara", "tiara", "meglio di là",
           "ascoltami", "sveltina"],
    "de": ["Klara", "hey Sara", "Schweiz", "schwelgen", "Svea", "hey Ara"],
    "en-us": ["Sarah", "tiara", "swell", "sweltering", "Laura", "Ara", "hey Ara",
              "hey Siri", "hey Google", "Alexa", "Aurora"],
}
CUSTOM_NEGATIVE_SHARE = 0.25        # share of the negative clips spent on the list above

# Native Piper voices (paths in huggingface.co/rhasspy/piper-voices) for a
# share of the clips; the rest come from the English multi-speaker generator.
PIPER_VOICES = [
    "it/it_IT/paola/medium/it_IT-paola-medium.onnx",
    "it/it_IT/riccardo/x_low/it_IT-riccardo-x_low.onnx",
]
PIPER_VOICE_SHARE = 0.5             # half of all clips from the Italian voices
PIPER_VOICE_UNIQUE = 0.2            # of those, the share actually synthesized (the rest are copies)

# Sizes, chosen to finish in about 1.5 h on a free T4 GPU.
N_SAMPLES = 25000                   # positive and (separately) negative training clips
N_SAMPLES_VAL = 2500                # positive and negative clips for validation
STEPS = 30000                       # training steps
FMA_HOURS = 1                       # hours of music used as background noise

## 2. Installation

Installs the speech generator, openWakeWord and the training libraries, and
writes four small helper files (the cells below this one). It does not change
Colab's PyTorch or NumPy; if an update ever forces it to, the cell says so and
asks you to restart the session.

In [ ]:
import importlib.metadata as md
import json, os, subprocess, sys, urllib.request

WORK = "/content/oww"
TOOLS, DATA, OUTPUT, VOICES = (f"{WORK}/{d}" for d in ("tools", "data", "output", "voices"))
OWW, PSG = f"{WORK}/openwakeword", f"{WORK}/piper-sample-generator"
for d in (TOOLS, DATA, OUTPUT, VOICES):
    os.makedirs(d, exist_ok=True)

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return
    print("downloading", url)
    tmp = path + ".part"
    urllib.request.urlretrieve(url, tmp)
    os.replace(tmp, path)

gpu = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                     capture_output=True, text=True).stdout.strip()
if gpu != "True":
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

def versions():
    out = {}
    for p in ("numpy", "torch"):
        try:
            out[p] = md.version(p)
        except md.PackageNotFoundError:
            out[p] = None
    return out

before = versions()

# Pinned where newer releases change the APIs train.py relies on. No TensorFlow:
# only the ONNX model is needed.
sh("pip install -q piper-tts==1.3.0 audiomentations==0.33.0 torch-audiomentations==0.12.0 "
   "speechbrain==1.1.1 acoustics==0.2.6 pronouncing==0.3.0 mutagen==1.48.1 torchinfo==1.8.0 "
   "torchmetrics==1.9.0 'onnx>=1.16' 'onnxscript>=0.2' soundfile pyyaml huggingface_hub")

if not os.path.isdir(OWW):
    sh(f"git clone -q --filter=blob:none https://github.com/dscripka/openwakeword {OWW}")
    sh(f"git -C {OWW} checkout -q 368c03716d1e92591906a84949bc477f3a834455")
sh(f"pip install -q --no-deps -e {OWW}")
if not os.path.isdir(PSG):
    sh(f"git clone -q --filter=blob:none https://github.com/rhasspy/piper-sample-generator {PSG}")
    sh(f"git -C {PSG} checkout -q 2971426a55072f7d22fec416ca7800df8bd23207")

models = f"{OWW}/openwakeword/resources/models"
os.makedirs(models, exist_ok=True)
for name in ("melspectrogram.onnx", "embedding_model.onnx"):
    fetch(f"https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{name}", f"{models}/{name}")
PT_MODEL = f"{PSG}/models/en_US-libritts_r-medium.pt"   # its .pt.json config ships in the repo
fetch("https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt", PT_MODEL)

VOICE_FILES = []
for rel in PIPER_VOICES:
    local = f"{VOICES}/{os.path.basename(rel)}"
    for suffix in ("", ".json"):
        fetch(f"https://huggingface.co/rhasspy/piper-voices/resolve/main/{rel}{suffix}", local + suffix)
    VOICE_FILES.append(local)

after = versions()
changed = [p for p in before if before[p] != after[p]]
if changed and any(p in sys.modules for p in changed):
    print("\n" + "!" * 70)
    print(f"{', '.join(changed)} changed version. Runtime → Restart session, then run all cells again (finished work is kept).")
    print("!" * 70)
else:
    print("\nInstallation done. Versions:", after)

### Helper files

Written to `/content/oww/tools/`; you do not need to read them.

- `oww_compat.py` patches the gaps between openWakeWord's training code and
  current libraries (torchaudio without `load`/`info`, SciPy without
  `sph_harm`, PyTorch's new ONNX exporter and pickle rules, a phonemizer model
  that is no longer downloadable).
- `generate_samples.py` is what `train.py` imports to make speech clips; it
  drives the current piper-sample-generator, mixes in the Piper voices, and
  resamples everything to 16 kHz.
- `run_train.py` runs `train.py` with those patches, keeps the model input at
  the stock 2 s window (16 frames) and saves the log.
- `fetch_data.py` downloads and converts the training data.

In [ ]:
%%writefile /content/oww/tools/oww_compat.py
# Compatibility patches so openWakeWord's train.py runs on current library
# versions. Imported by run_train.py (and by the sample-generator shim) before
# anything else.
import os
import types

# deep-phonemizer (used by openWakeWord for words missing from the CMU
# dictionary) and the piper generator checkpoint are full pickles; PyTorch
# >= 2.6 refuses them unless weights_only loading is switched off.
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

# acoustics 0.2.6 imports scipy.special.sph_harm, removed in SciPy 1.17.
import scipy.special  # noqa: E402

if not hasattr(scipy.special, "sph_harm"):
    def _sph_harm(m, n, theta, phi):
        return scipy.special.sph_harm_y(n, m, phi, theta)
    scipy.special.sph_harm = _sph_harm

# torchaudio >= 2.9 routes load() through torchcodec and dropped info();
# openWakeWord and torch-audiomentations only read WAV/FLAC files, which
# soundfile handles directly.
import soundfile  # noqa: E402
import torch  # noqa: E402
import torchaudio  # noqa: E402


def _ta_load(uri, frame_offset=0, num_frames=-1, normalize=True,
             channels_first=True, format=None, buffer_size=4096, backend=None):
    frames = num_frames if num_frames is not None and num_frames > 0 else -1
    data, sr = soundfile.read(str(uri), start=frame_offset or 0, frames=frames,
                              dtype="float32", always_2d=True)
    tensor = torch.from_numpy(data.T.copy() if channels_first else data.copy())
    return tensor, sr


def _ta_info(uri, *args, **kwargs):
    i = soundfile.info(str(uri))
    return types.SimpleNamespace(sample_rate=i.samplerate, num_frames=i.frames,
                                 num_channels=i.channels, bits_per_sample=16,
                                 encoding="PCM_S")


torchaudio.load = _ta_load
torchaudio.info = _ta_info

# openWakeWord looks up words missing from the CMU dictionary with a
# DeepPhonemizer model whose download (an S3 bucket) has been disabled. Fall
# back to espeak-ng (bundled with piper-tts) and map its IPA to ARPAbet, and
# return a single pronunciation per word (several break train.py's
# word/pronunciation pairing).
import pronouncing  # noqa: E402

_IPA_ARPA = [("aɪ", "AY"), ("aʊ", "AW"), ("eɪ", "EY"), ("oʊ", "OW"), ("ɔɪ", "OY"),
             ("tʃ", "CH"), ("dʒ", "JH"), ("ɑː", "AA"), ("ɜː", "ER"), ("iː", "IY"),
             ("uː", "UW"), ("ɔː", "AO"), ("ɝ", "ER"), ("ɚ", "ER"), ("ɑ", "AA"),
             ("æ", "AE"), ("ʌ", "AH"), ("ə", "AH"), ("ɐ", "AH"), ("ɔ", "AO"),
             ("ɛ", "EH"), ("ɪ", "IH"), ("ᵻ", "IH"), ("ɨ", "IH"), ("ɜ", "ER"), ("ɒ", "AA"), ("i", "IY"), ("ʊ", "UH"), ("u", "UW"),
             ("a", "AA"), ("e", "EH"), ("o", "OW"), ("b", "B"), ("d", "D"),
             ("ð", "DH"), ("f", "F"), ("ɡ", "G"), ("g", "G"), ("h", "HH"),
             ("k", "K"), ("l", "L"), ("m", "M"), ("n", "N"), ("ŋ", "NG"),
             ("p", "P"), ("ɹ", "R"), ("r", "R"), ("s", "S"), ("ʃ", "SH"),
             ("t", "T"), ("ɾ", "T"), ("θ", "TH"), ("v", "V"), ("w", "W"),
             ("j", "Y"), ("z", "Z"), ("ʒ", "ZH")]
_ARPA_VOWELS = {"AY", "AW", "EY", "OW", "OY", "AA", "ER", "IY", "UW", "AO",
                "AE", "AH", "EH", "IH", "UH"}
_orig_phones_for_word = pronouncing.phones_for_word
_espeak = None


def _espeak_arpabet(word):
    global _espeak
    if _espeak is None:
        from piper.phonemize_espeak import EspeakPhonemizer
        _espeak = EspeakPhonemizer()
    ipa = "".join(p for s in _espeak.phonemize("en-us", word) for p in s)
    out, stress, i = [], "0", 0
    while i < len(ipa):
        if ipa[i] in "ˈˌ":
            stress = "1" if ipa[i] == "ˈ" else "2"
            i += 1
            continue
        for sym, arpa in _IPA_ARPA:
            if ipa.startswith(sym, i):
                if arpa in _ARPA_VOWELS:
                    arpa, stress = arpa + stress, "0"
                out.append(arpa)
                i += len(sym)
                break
        else:
            i += 1  # length marks, punctuation, anything unmapped
    return " ".join(out)


def _phones_for_word(word):
    phones = _orig_phones_for_word(word)
    if phones:
        return phones[:1]
    arpa = _espeak_arpabet(word)
    return [arpa] if arpa else []


pronouncing.phones_for_word = _phones_for_word

# PyTorch >= 2.9 exports through the dynamo exporter by default, which
# ignores opset_version=13 and needs onnxscript. The classic exporter gives
# the small opset-13 graph with the fixed [1, 16, 96] input that the stock
# openWakeWord models have; use it while this PyTorch still ships it.
import inspect  # noqa: E402

_orig_onnx_export = torch.onnx.export
if "dynamo" in inspect.signature(_orig_onnx_export).parameters:
    def _onnx_export(*args, **kwargs):
        if "dynamo" in kwargs:
            return _orig_onnx_export(*args, **kwargs)
        try:
            return _orig_onnx_export(*args, dynamo=False, **kwargs)
        except Exception as exc:  # classic exporter missing or broken
            import logging
            logging.warning("Classic ONNX export failed (%s); using the dynamo exporter", exc)
            return _orig_onnx_export(*args, dynamo=True, **kwargs)
    torch.onnx.export = _onnx_export

In [ ]:
%%writefile /content/oww/tools/generate_samples.py
# Adapter between openWakeWord's train.py and the current
# piper-sample-generator package.
#
# train.py does `from generate_samples import generate_samples` and calls it
# without a model argument, expecting 16 kHz WAV files. This module keeps that
# interface and dispatches to the package: the English multi-speaker
# generator (.pt) for one share of the clips, Piper voices (.onnx) for the
# rest, then resamples every clip to 16 kHz (openWakeWord rejects any other
# rate). Settings come from shim_config.json next to this file.
import itertools
import json
import logging
import os
import random
import shutil
import sys
import unicodedata
import uuid
import wave
from math import gcd
from pathlib import Path

import numpy as np
from scipy.signal import resample_poly

_HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(_HERE))
import oww_compat  # noqa: E402,F401

with open(_HERE / "shim_config.json", encoding="utf-8") as _f:
    CFG = json.load(_f)

# piper_train (needed to unpickle the .pt generator) exists only in the git
# checkout of piper-sample-generator, not in the wheel.
sys.path.insert(0, CFG["psg_repo"])
import piper_sample_generator.__main__ as psg  # noqa: E402

_LOGGER = logging.getLogger("generate_samples_shim")
TARGET_SR = 16000


class _ShuffledItertools:
    """Stands in for itertools inside the package: product() is shuffled, so
    every call draws random speaker pairs from all 904 speakers instead of
    walking (0, 0), (0, 1), ... from the start each time."""
    cycle = itertools.cycle
    islice = itertools.islice

    @staticmethod
    def product(*iterables):
        items = list(itertools.product(*iterables))
        random.shuffle(items)
        return items


psg.it = _ShuffledItertools

# Phonemes the English generator never saw in training, mapped to the
# closest English ones (used for texts phonemized in another language).
# Diphthongs English has are kept as they are; longest match wins.
_SUBST = [("aɪ", "aɪ"), ("aʊ", "aʊ"), ("ɔʏ", "ɔɪ"), ("ʎ", "lj"), ("ɲ", "nj"),
          ("r", "ɾ"), ("ʁ", "ɹ"), ("ʀ", "ɹ"), ("ç", "ʃ"), ("x", "h"), ("χ", "h"),
          ("ɣ", "ɡ"), ("β", "b"), ("ʏ", "ʊ"), ("y", "i"), ("ø", "ɜ"), ("œ", "ɜ"),
          ("e", "ɛ"), ("o", "ɔ"), ("a", "ɑ")]
_VOWELS = set("aeiouɑɐɒæɛɜəɪʊɔʌyøœɨʉ")


def _norm(text):
    return " ".join(text.lower().split())


def _to_english_phonemes(text):
    """Phoneme string for the English generator. A text listed in
    pt_text_languages is phonemized with that language's espeak voice and
    mapped onto English phonemes, so English voices say it the foreign way."""
    lang = CFG.get("pt_text_languages", {}).get(_norm(text), "en-us")
    sentences = psg._PHONEMIZER.phonemize(lang, text)
    ipa = "".join(p for s in sentences for p in s)
    if lang.startswith("en"):
        return ipa
    ipa = unicodedata.normalize("NFC", ipa)  # so that ç is one character
    out, i = "", 0
    while i < len(ipa):
        ch = ipa[i]
        if ch == "ː" and (not out or out[-1] not in _VOWELS):
            i += 1  # geminate consonants: English has no long consonants
            continue
        for src, dst in _SUBST:
            if ipa.startswith(src, i):
                out += dst
                i += len(src)
                break
        else:
            out += ch
            i += 1
    return out


def _as_list(text):
    if isinstance(text, str):
        if os.path.isfile(text):
            with open(text, encoding="utf-8") as f:
                return [line.strip() for line in f if line.strip()]
        return [text]
    return list(text)


def _move_resampled(staging, output_dir):
    n = 0
    for wav_path in sorted(staging.glob("*.wav")):
        with wave.open(str(wav_path), "rb") as w:
            sr = w.getframerate()
            audio = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
        if audio.size == 0:
            wav_path.unlink()
            continue
        if sr != TARGET_SR:
            g = gcd(sr, TARGET_SR)
            audio = resample_poly(audio.astype(np.float32), TARGET_SR // g, sr // g)
            audio = np.clip(audio, -32768, 32767).astype(np.int16)
        with wave.open(str(output_dir / wav_path.name), "wb") as w:
            w.setnchannels(1)
            w.setsampwidth(2)
            w.setframerate(TARGET_SR)
            w.writeframes(audio.tobytes())
        wav_path.unlink()
        n += 1
    return n


def _is_oom(exc):
    return "out of memory" in str(exc).lower()


def generate_samples(text, max_samples, output_dir, file_names=None,
                     batch_size=1, noise_scales=(0.667,), noise_scale_ws=(0.8,),
                     length_scales=(0.75, 1.0, 1.25), **kwargs):
    import torch

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    staging = Path(str(output_dir).rstrip("/") + "__staging")
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)

    max_samples = int(max_samples)
    # Shuffled, then consumed in order across chunks: every text is used about
    # equally often, and repeated custom negatives stay spread over the run.
    texts = _as_list(text)
    random.shuffle(texts)

    def text_slice(start, n):
        return [texts[(start + i) % len(texts)] for i in range(n)]
    names = list(file_names)[:max_samples] if file_names else []
    names += [uuid.uuid4().hex + ".wav" for _ in range(max_samples - len(names))]

    onnx_models = CFG.get("onnx_models") or []
    n_onnx = int(round(max_samples * CFG.get("onnx_share", 0.0))) if onnx_models else 0
    n_pt = max_samples - n_onnx
    pt_names, onnx_names = names[:n_pt], names[n_pt:]
    n_unique = 0
    written = 0

    if n_pt > 0:
        cache = {}

        def phonemes_for(chunk_texts):
            for t in chunk_texts:
                if t not in cache:
                    cache[t] = _to_english_phonemes(t)
            return [cache[t] for t in chunk_texts]

        _LOGGER.info("English generator: %d clips, e.g. %r -> %r", n_pt, texts[0], phonemes_for(texts[:1])[0])
        bs = max(1, int(batch_size))
        chunk = int(CFG.get("pt_chunk", 2000))
        done = 0
        while done < n_pt:
            todo = pt_names[done:done + chunk]
            try:
                psg.generate_samples(text=phonemes_for(text_slice(done, len(todo))), output_dir=staging, model=CFG["pt_model"],
                                     max_samples=len(todo), file_names=todo, batch_size=bs,
                                     noise_scales=list(noise_scales), noise_scale_ws=list(noise_scale_ws),
                                     length_scales=list(length_scales), phoneme_input=True)
            except (RuntimeError, torch.cuda.OutOfMemoryError) as exc:
                if not _is_oom(exc) or bs == 1:
                    raise
                bs = max(1, bs // 2)
                _LOGGER.warning("CUDA out of memory, retrying with batch size %d", bs)
                for p in staging.glob("*.wav"):
                    p.unlink()
                torch.cuda.empty_cache()
                continue
            written += _move_resampled(staging, output_dir)
            done += len(todo)
            _LOGGER.info("English generator: %d/%d clips", done, n_pt)

    if n_onnx > 0:
        # Piper voices run on the CPU (onnxruntime), roughly 5-10x slower than
        # the GPU generator, and a handful of voices adds little variety
        # after a few thousand clips. So only a share of the clips is
        # synthesized; the rest are copies, which the augmentation step
        # (noise, reverb, pitch, EQ) turns into different training examples.
        frac = float(CFG.get("onnx_unique_fraction", 1.0))
        n_unique = max(1, min(n_onnx, int(round(n_onnx * frac))))
        _LOGGER.info("Piper voices %s: %d clips (%d synthesized, the rest copies)",
                     [Path(m).name for m in onnx_models], n_onnx, n_unique)
        chunk = int(CFG.get("onnx_chunk", 500))
        done = 0
        while done < n_unique:
            todo = onnx_names[done:min(done + chunk, n_unique)]
            psg.generate_samples_onnx(text=text_slice(n_pt + done, len(todo)), output_dir=staging, model=onnx_models,
                                      max_samples=len(todo), file_names=todo,
                                      length_scales=CFG.get("onnx_length_scales", list(length_scales)),
                                      noise_scales=CFG.get("onnx_noise_scales", list(noise_scales)),
                                      noise_scale_ws=CFG.get("onnx_noise_scale_ws", list(noise_scale_ws)))
            written += _move_resampled(staging, output_dir)
            done += len(todo)
            _LOGGER.info("Piper voices: %d/%d clips synthesized", done, n_unique)
        sources = [n for n in onnx_names[:n_unique] if (output_dir / n).exists()]
        for name in onnx_names[n_unique:]:
            if sources:
                shutil.copyfile(output_dir / random.choice(sources), output_dir / name)
                written += 1

    # Which engine made which clip, for the listening cell in the notebook.
    with open(str(output_dir).rstrip("/") + "__sources.tsv", "a", encoding="utf-8") as f:
        for i, name in enumerate(names):
            kind = "english" if i < n_pt else ("piper" if i < n_pt + n_unique else "piper-copy")
            f.write(f"{name}\t{kind}\n")

    shutil.rmtree(staging, ignore_errors=True)
    _LOGGER.info("Wrote %d clips to %s", written, output_dir)

In [ ]:
%%writefile /content/oww/tools/run_train.py
# Runs openWakeWord's train.py with the compatibility patches applied and
# its log copied to a file.
# Usage: python run_train.py <train.py> <log file> --training_config ... [--generate_clips ...]
import logging
import os
import runpy
import sys
import warnings

# Library chatter (deprecation notices, docstring escapes in acoustics) that
# would bury the progress output in Colab.
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*dtype is np.float64.*")
warnings.filterwarnings("ignore", message=".*TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD.*")

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import oww_compat  # noqa: E402,F401

train_py, log_file = sys.argv[1], sys.argv[2]
del sys.argv[1:3]

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout),
                              logging.FileHandler(log_file, encoding="utf-8")])

with open(train_py, encoding="utf-8") as f:
    source = f.read()
# train.py declares --convert_to_tflite with default="False" (a non-empty, so
# truthy, string) and then tests `if args.convert_to_tflite:`, so it always
# tries the TensorFlow conversion. Only the ONNX model is wanted here.
source = source.replace("if args.convert_to_tflite:", "if args.convert_to_tflite is True:")
# train.py sizes the training window from the median positive clip and, for
# a phrase longer than ~1.4 s, makes it longer than the stock 2 s (16
# embedding frames). "total_length_override" in the config pins it, so the
# model keeps the [1, 16, 96] input of the stock openWakeWord models.
anchor = ('''    elif abs(config["total_length"] - 32000) <= 4000:
        config["total_length"] = 32000
''')
if anchor not in source:
    raise SystemExit("train.py changed: cannot apply total_length_override")
source = source.replace(anchor, anchor + '''    if config.get("total_length_override"):
        if config["total_length"] > config["total_length_override"]:
            logging.warning(f"Positive clips are long (window would be {config['total_length']} samples); "
                            f"using {config['total_length_override']}")
        config["total_length"] = config["total_length_override"]
''', 1)
patched = os.path.join(os.path.dirname(os.path.abspath(log_file)), "train_patched.py")
with open(patched, "w", encoding="utf-8") as f:
    f.write(source)

sys.argv[0] = patched
runpy.run_path(patched, run_name="__main__")

In [ ]:
%%writefile /content/oww/tools/fetch_data.py
# Downloads the training data openWakeWord's recipe uses and converts all
# audio to 16 kHz mono 16-bit WAV. Every part is skipped when already done,
# so the cell can be re-run after an interruption.
# Usage: python fetch_data.py <data dir> <hours of FMA music>
import glob
import io
import os
import shutil
import sys
import tarfile
from math import gcd

import numpy as np
import soundfile as sf
from huggingface_hub import hf_hub_download, list_repo_files, snapshot_download
from scipy.signal import resample_poly
from tqdm import tqdm

DATA = sys.argv[1]
FMA_HOURS = float(sys.argv[2]) if len(sys.argv) > 2 else 1.0
AUDIO_EXT = (".wav", ".flac", ".mp3", ".ogg")
os.makedirs(DATA, exist_ok=True)


def write_16k(path, data, sr):
    data = np.asarray(data, dtype=np.float32)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != 16000:
        g = gcd(int(sr), 16000)
        data = resample_poly(data, 16000 // g, int(sr) // g)
    peak = float(np.max(np.abs(data))) if data.size else 0.0
    if peak > 1.0:
        data = data / peak
    sf.write(path, (data * 32767).astype(np.int16), 16000, subtype="PCM_16")


def read_any(src):
    """soundfile reads WAV/FLAC/MP3/OGG; librosa is the fallback."""
    try:
        return sf.read(src, dtype="float32", always_2d=False)
    except Exception:
        import librosa
        if isinstance(src, io.BytesIO):
            src.seek(0)
        data, sr = librosa.load(src, sr=None, mono=True)
        return data, sr


def convert_files(files, out_dir, limit=None):
    os.makedirs(out_dir, exist_ok=True)
    n = 0
    for f in tqdm(files[:limit] if limit else files, desc=os.path.basename(out_dir)):
        name = os.path.splitext(os.path.basename(f))[0] + ".wav"
        try:
            data, sr = read_any(f)
            write_16k(os.path.join(out_dir, name), data, sr)
            n += 1
        except Exception as exc:
            print(f"  skipped {f}: {exc}")
    return n


def done(out_dir, minimum):
    return os.path.isdir(out_dir) and len(os.listdir(out_dir)) >= minimum


def via_datasets(repo, out_dir, name=None, limit=None):
    """Streams a dataset without letting `datasets` decode the audio (recent
    versions need torchcodec for that) and decodes the bytes here."""
    import datasets
    ds = datasets.load_dataset(repo, name=name, split="train", streaming=True)
    ds = ds.cast_column("audio", datasets.Audio(decode=False))
    os.makedirs(out_dir, exist_ok=True)
    n = 0
    for row in tqdm(ds, desc=os.path.basename(out_dir), total=limit):
        audio = row["audio"]
        raw = audio.get("bytes")
        if raw is None:
            import fsspec
            with fsspec.open(audio["path"], "rb") as f:
                raw = f.read()
        base = os.path.splitext(os.path.basename(audio.get("path") or f"clip{n}"))[0]
        try:
            data, sr = read_any(io.BytesIO(raw))
            write_16k(os.path.join(out_dir, f"{base}.wav"), data, sr)
            n += 1
        except Exception as exc:
            print(f"  skipped {base}: {exc}")
        if limit and n >= limit:
            break
    return n


# 1. Room impulse responses (MIT survey, 270 files).
rir_dir = os.path.join(DATA, "mit_rirs")
if not done(rir_dir, 200):
    repo = "davidscripka/MIT_environmental_impulse_responses"
    raw = snapshot_download(repo, repo_type="dataset", local_dir=os.path.join(DATA, "raw_rirs"),
                            allow_patterns=["*.wav", "*/*.wav", "*/*/*.wav", "*.flac", "*/*.flac"])
    files = sorted(glob.glob(os.path.join(raw, "**", "*.*"), recursive=True))
    files = [f for f in files if f.lower().endswith(AUDIO_EXT)]
    n = convert_files(files, rir_dir) if files else via_datasets(repo, rir_dir)
    print(f"Room impulse responses: {n}")
else:
    print("Room impulse responses: already there")

# 2. AudioSet: one part of the balanced training set (~10 s clips).
def audioset_part(fname):
    out_dir = os.path.join(DATA, "audioset_16k")
    marker = os.path.join(out_dir, f".done_{fname}")
    if os.path.exists(marker):
        print(f"AudioSet {fname}: already there")
        return
    repo = "agkphysics/AudioSet"
    try:
        tar_path = hf_hub_download(repo, f"data/{fname}", repo_type="dataset",
                                   local_dir=os.path.join(DATA, "raw_audioset"))
    except Exception:  # the file moved: look for it by name
        matches = [f for f in list_repo_files(repo, repo_type="dataset") if f.endswith(fname)]
        if not matches:
            raise
        tar_path = hf_hub_download(repo, matches[0], repo_type="dataset",
                                   local_dir=os.path.join(DATA, "raw_audioset"))
    extract_dir = os.path.join(DATA, "raw_audioset", fname + "_x")
    with tarfile.open(tar_path) as tar:
        try:
            tar.extractall(extract_dir, filter="data")
        except TypeError:  # Python without extraction filters
            tar.extractall(extract_dir)
    files = sorted(glob.glob(os.path.join(extract_dir, "**", "*.flac"), recursive=True))
    n = convert_files(files, out_dir)
    shutil.rmtree(extract_dir, ignore_errors=True)
    os.remove(tar_path)
    open(marker, "w").close()
    print(f"AudioSet {fname}: {n} clips")


try:
    audioset_part("bal_train09.tar")
except Exception as exc:
    print(f"AudioSet download failed ({exc}); continuing with the music clips only.")

# 3. Free Music Archive: FMA_HOURS of 30 s music clips.
fma_dir = os.path.join(DATA, "fma")
n_fma = int(FMA_HOURS * 3600 // 30)
if n_fma > 0 and not done(fma_dir, int(n_fma * 0.9)):
    try:
        repo = "rudraml/fma"
        files = [f for f in list_repo_files(repo, repo_type="dataset") if f.lower().endswith(AUDIO_EXT)]
        small = [f for f in files if "small" in f.lower()] or files
        if small:
            local = [hf_hub_download(repo, f, repo_type="dataset", local_dir=os.path.join(DATA, "raw_fma"))
                     for f in tqdm(small[:n_fma], desc="FMA download")]
            n = convert_files(local, fma_dir)
        else:
            n = via_datasets(repo, fma_dir, name="small", limit=n_fma)
        print(f"FMA music clips: {n}")
    except Exception as exc:
        # Older layouts of this dataset need a loading script, which
        # datasets >= 4 no longer runs. Music is a nice-to-have: use a
        # second AudioSet part instead.
        print(f"FMA download failed ({exc}); using a second AudioSet part instead.")
        try:
            audioset_part("bal_train08.tar")
        except Exception as exc2:
            print(f"AudioSet download failed too ({exc2}).")
else:
    print("FMA music clips: already there (or not wanted)")

# 4. Precomputed openWakeWord features: ~2,000 h of negative speech/noise/music
# (ACAV100M, ~17 GB) and ~11 h for measuring false positives per hour.
for fname in ["openwakeword_features_ACAV100M_2000_hrs_16bit.npy", "validation_set_features.npy"]:
    path = os.path.join(DATA, fname)
    if not os.path.exists(path):
        print(f"Downloading {fname} ...")
        hf_hub_download("davidscripka/openwakeword_features", fname, repo_type="dataset", local_dir=DATA)
    print(f"{fname}: {os.path.getsize(path) / 1e9:.1f} GB")

backgrounds = [d for d in ("audioset_16k", "fma") if os.path.isdir(os.path.join(DATA, d)) and os.listdir(os.path.join(DATA, d))]
print("Background folders:", backgrounds)
if not backgrounds:
    raise SystemExit("No background audio could be downloaded; see the errors above.")
with open(os.path.join(DATA, "background_dirs.txt"), "w") as f:
    f.write("\n".join(os.path.join(DATA, d) for d in backgrounds))

## 3. Download the training data

Room impulse responses (MIT), one part of AudioSet and an hour of the Free
Music Archive as background noise, and openWakeWord's precomputed features:
~2,000 hours of speech, noise and music that must not wake the model (about
17 GB), plus ~11 hours used to count false wakes per hour. If the music
download fails, a second AudioSet part takes its place.

In [ ]:
!{sys.executable} {TOOLS}/fetch_data.py {DATA} {FMA_HOURS}

for need in ("mit_rirs", "background_dirs.txt", "openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
             "validation_set_features.npy"):
    assert os.path.exists(f"{DATA}/{need}"), f"{need} is missing: look for the error above and run this cell again."

### Write the training configuration

From the settings in step 1 and the data found in step 3.

In [ ]:
import json

def norm(text):
    return " ".join(text.lower().split())

# Texts the English generator should speak in another language than English.
text_languages = {norm(t): lang for lang, texts in CUSTOM_NEGATIVES.items()
                  if not lang.startswith("en") for t in texts}
if not PHRASE_LANGUAGE.startswith("en"):
    text_languages[norm(TARGET_PHRASE)] = PHRASE_LANGUAGE

with open(f"{TOOLS}/shim_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "psg_repo": PSG,
        "pt_model": PT_MODEL,
        "pt_text_languages": text_languages,
        "onnx_models": VOICE_FILES,
        "onnx_share": PIPER_VOICE_SHARE if VOICE_FILES else 0.0,
        "onnx_unique_fraction": PIPER_VOICE_UNIQUE,
        "onnx_length_scales": [0.8, 0.9, 1.0, 1.1, 1.2],
        "onnx_noise_scales": [0.5, 0.667, 0.8, 0.98],
        "onnx_noise_scale_ws": [0.6, 0.8, 1.0],
    }, f, ensure_ascii=False, indent=1)

negatives = [t for texts in CUSTOM_NEGATIVES.values() for t in texts]
repeat = max(1, round(CUSTOM_NEGATIVE_SHARE * N_SAMPLES / max(1, len(negatives))))

if os.path.exists(f"{DATA}/background_dirs.txt"):
    with open(f"{DATA}/background_dirs.txt") as f:
        backgrounds = [line.strip() for line in f if line.strip()]
else:
    backgrounds = [f"{DATA}/audioset_16k", f"{DATA}/fma"]

config = {
    "model_name": MODEL_NAME,
    "target_phrase": [TARGET_PHRASE],
    "custom_negative_phrases": negatives * repeat,
    "n_samples": N_SAMPLES,
    "n_samples_val": N_SAMPLES_VAL,
    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "piper_sample_generator_path": TOOLS,
    "output_dir": OUTPUT,
    "rir_paths": [f"{DATA}/mit_rirs"],
    "background_paths": backgrounds,
    "background_paths_duplication_rate": [1] * len(backgrounds),
    "false_positive_validation_data_path": f"{DATA}/validation_set_features.npy",
    "augmentation_rounds": 1,
    "feature_data_files": {"ACAV100M_sample": f"{DATA}/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
    "model_type": "dnn",
    "layer_size": 32,
    "steps": STEPS,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": 0.2,
    "total_length_override": 32000,   # 2 s window = 16 frames, the input size of stock models
}
CONFIG_FILE = f"{WORK}/{MODEL_NAME}.yaml"
with open(CONFIG_FILE, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=1)   # JSON is valid YAML

LOG = f"{WORK}/{MODEL_NAME}_train.log"
TRAIN = f"{sys.executable} {TOOLS}/run_train.py {OWW}/openwakeword/train.py {LOG} --training_config {CONFIG_FILE}"
CLIPS = f"{OUTPUT}/{MODEL_NAME}"
print("Training config written to", CONFIG_FILE)

## 4. Generate speech clips

Positive clips (the phrase) and negative clips (the list from step 1 plus
automatically found sound-alikes), each for training and for validation. Re-run
this cell if it is interrupted; it continues where it stopped.

In [ ]:
!{TRAIN} --generate_clips

for d, n in (("positive_train", N_SAMPLES), ("negative_train", N_SAMPLES),
             ("positive_test", N_SAMPLES_VAL), ("negative_test", N_SAMPLES_VAL)):
    have = len(os.listdir(f"{CLIPS}/{d}")) if os.path.isdir(f"{CLIPS}/{d}") else 0
    print(f"{d}: {have} clips")
    assert have >= 0.95 * n, f"{d} is incomplete: look for the error above and run this cell again."

### Listen before training

Eight random positive clips. They should all clearly say the phrase; clips from
the English voices may sound accented, but the words must be recognizable. If
most of them are wrong, stop here: training on them would give a useless model.

In [ ]:
import random
from IPython.display import Audio, display

sources = {}
for d in ("positive_train", "positive_test"):
    tsv = f"{CLIPS}/{d}__sources.tsv"
    if os.path.exists(tsv):
        with open(tsv) as f:
            sources.update(line.rstrip("\n").split("\t") for line in f if "\t" in line)

clips = [f"{CLIPS}/positive_train/{n}" for n in os.listdir(f"{CLIPS}/positive_train")]
labels = {"english": "English multi-speaker voices", "piper": "native Piper voice",
          "piper-copy": "native Piper voice"}
for path in random.sample(clips, min(8, len(clips))):
    print(labels.get(sources.get(os.path.basename(path)), "generated"))
    display(Audio(filename=path))

## 5. Augment the clips and compute features

Mixes every clip with background noise and room reverb, then converts it into
the 16×96 openWakeWord features the model is trained on.

In [ ]:
!{TRAIN} --augment_clips

for f in ("positive_features_train", "negative_features_train", "positive_features_test", "negative_features_test"):
    assert os.path.exists(f"{CLIPS}/{f}.npy"), f"{f}.npy is missing: look for the error above and run this cell again."
print("Features ready.")

## 6. Train

Three training rounds with decreasing learning rate; when the model wakes too
often on the 11-hour validation audio, the next round weighs negatives more.
The best checkpoints are averaged into the final model.

In [ ]:
if os.path.exists(f"{OUTPUT}/{MODEL_NAME}.onnx"):
    print(f"Already trained. To train again, delete {OUTPUT}/{MODEL_NAME}.onnx and run this cell.")
else:
    !{TRAIN} --train_model

assert os.path.exists(f"{OUTPUT}/{MODEL_NAME}.onnx"), "No model was written: look for the error above."

## 7. Check the result and download it

The two numbers that matter, both measured on data the model did not train on:

- **Recall**: the share of spoken phrases it detects. 0.7 or more is good; the
  synthetic validation clips are harder than a real voice close to the phone,
  so real-life recall is usually higher.
- **False positives per hour**: how often it wakes by mistake in 11 hours of
  ordinary speech, music and noise. 0.5 or less is good (one false wake every
  two hours or less).

Both depend on the detection threshold (0.5 here); the shell's sensitivity
slider moves that trade-off later.

In [ ]:
import re

with open(LOG, encoding="utf-8") as f:
    log = f.read()

def last(label):
    found = re.findall(label + r":\s*([0-9.eE+-]+)", log)
    return float(found[-1]) if found else None

recall, fp_hour, accuracy = last("Final Model Recall"), last("Final Model False Positives per Hour"), last("Final Model Accuracy")
print(f"Recall: {recall}   False positives per hour: {fp_hour}   Accuracy: {accuracy}")
if recall is not None and fp_hour is not None:
    if recall >= 0.7 and fp_hour <= 0.5:
        print("Good: detects most phrases and rarely wakes by mistake.")
    if recall < 0.7:
        print("Misses many phrases. Try it anyway (a real voice near the phone is easier than the synthetic "
              "test clips) with a lower threshold; for a better model, train again with more N_SAMPLES or STEPS.")
    if fp_hour > 0.5:
        print("Wakes too often by mistake. Usable with a higher threshold in the shell; "
              "for a better model, train again with more STEPS or more N_SAMPLES.")

MODEL_FILE = f"{OUTPUT}/{MODEL_NAME}.onnx"
check = subprocess.run([sys.executable, "-c",
    "import onnxruntime as ort, sys; s = ort.InferenceSession(sys.argv[1]); "
    "print([i.shape for i in s.get_inputs()], [o.shape for o in s.get_outputs()])", MODEL_FILE],
    capture_output=True, text=True)
print("Model input and output shapes:", check.stdout.strip() or check.stderr.strip())

from google.colab import files
files.download(MODEL_FILE)

## 8. Use it in the Ara shell

1. Copy `svegliati_ara.onnx` to the phone (it lands in your browser's downloads;
   e.g. save it to Drive or send it to yourself).
2. In the Ara shell open **Shell settings** (the gear button next to the app
   drawer's search field, or long-press the launcher bar) and scroll to
   **Wake phrase**.
3. Under **Models**, tap **Download feature models** if the melspectrogram and
   embedding models still show as missing, then tap **Choose file…** and pick
   `svegliati_ara.onnx`. The models line should now read *Phrase: present*.
4. Choose when to listen (**While the home screen is visible** or **Always**)
   and say “Svegliati ara”.
5. Tune **Sensitivity threshold** with the slider: lower wakes more easily (and
   more often by mistake), higher wakes less often; 0.5 is openWakeWord's
   default. If it misses you, try 0.3–0.4; if it wakes by itself, try 0.6–0.7.